# Kepler-51 planetary system in the light of the PhotoRing effect
## Reproducing the ring-retrieval summary tables of Numpaque et al. (2026)

This notebook contains the code for reproducing the tables in the paper:

> S. Numpaque, J.I. Zuluaga, D. Kipping and J.A. Alvarado "Asterodensity profile analysis of the Kepler-51 planetary system:
a case study on the detection of exoplanetary rings via the PhotoRing effect", in preparation.

For detailed explanations see the GitHub repository: [https://github.com/seap-udea/PRisma](https://github.com/seap-udea/PRisma).

**Outputs generated by this notebook:**
- `tab_full_set.tex` — Full-set retrieval results table
- `tab_grid_golden.tex` — Golden Sample grid-search results table

## Preparing the environment

In [1]:
# ── Bootstrap: make the repository packages importable without installation ──
import sys, pathlib
_HERE = pathlib.Path.cwd()
_cands = [_HERE, *_HERE.parents]
_REPO = next((c for c in _cands if (c / "pipeline" / "photoring").is_dir() and (c / "pipeline" / "exorings").is_dir()), _HERE.parent)
for _p in (str(_REPO / "pipeline"), str(_REPO)):
    if _p not in sys.path:
        sys.path.insert(0, _p)
print("repo root:", _REPO)

repo root: /Users/jzuluaga/dev/PRisma


In [2]:
import json
import re
import numpy as np
from pathlib import Path

import photoring as pr

CASE = "kepler_51"
paths = pr.CasePaths(CASE, pipeline_dir=_REPO / "pipeline")
RESULTS = paths.results_dir("exorings")
FULL_RESULTS = RESULTS / "full_masuda"
EXPLORE_RESULTS = RESULTS / "explore_radius_alpha_masuda"
IMG = pathlib.Path(_REPO / "papers" / "kepler51" / "figures")
IMG.mkdir(parents=True, exist_ok=True)
print("full-set results:", FULL_RESULTS)
print("grid-search results:", EXPLORE_RESULTS)

full-set results: /Users/jzuluaga/dev/PRisma/pipeline/kepler_51/results/exorings/full_masuda
grid-search results: /Users/jzuluaga/dev/PRisma/pipeline/kepler_51/results/exorings/explore_radius_alpha_masuda


## Physical constants and helpers

In [3]:
M_E_M_S = 3.0034895e-6  # Earth mass in solar masses
R_E_R_S = 0.0091577     # Earth radius in solar radii
RHO_E = 5.51            # Earth density g/cm^3
M_p_earth = 6.9         # Earth masses (adopted for both planets)
R_star_sun = 0.869      # Solar radii (Berger et al. 2023)
R_star_earth = R_star_sun / R_E_R_S

PLANETS = ["b", "d"]

def get_stats(data, param):
    med = data[f"stat_{param}_median"]
    p16 = data[f"stat_{param}_p16"]
    p84 = data[f"stat_{param}_p84"]
    upper = p84 - med
    lower = med - p16
    return med, upper, lower

def format_val(val, up, low, fmt="{:.2f}"):
    s_val = fmt.format(val)
    s_up = fmt.format(up)
    s_low = fmt.format(low)
    return f"${s_val}^{{+{s_up}}}_{{-{s_low}}}$"

def get_derived(data):
    med_p, _, _ = get_stats(data, "p")
    R_p_earth = med_p * R_star_earth
    rho_p = RHO_E * M_p_earth / (R_p_earth**3)
    return R_p_earth, rho_p

from photoring.io import fixed_p_from_meta

def get_param(data, param, run_tag="", fmt="{:.2f}"):
    if f"stat_{param}_median" in data:
        med = data[f"stat_{param}_median"]
        p16 = data[f"stat_{param}_p16"]
        p84 = data[f"stat_{param}_p84"]
        upper = p84 - med
        lower = med - p16
        return f"${fmt.format(med)}^{{+{fmt.format(upper)}}}_{{-{fmt.format(lower)}}}$", med
    elif param == "p" and not data.get("P_FREE", True):
        val = fixed_p_from_meta(data)
        return f"${fmt.format(val)}$", val
    elif f"{param.upper()}_FIXED" in data:
        val = data[f"{param.upper()}_FIXED"]
        return f"${fmt.format(val)}$", val
    else:
        if param == "p" and "_P" in run_tag:
            m = re.search(r"_P([13579])(?:_|$)", run_tag)
            if m:
                frac = float(m.group(1)) / 10.0
                p_min = data.get("p_min", 0.0)
                p_max = data.get("p_mean_ref", 0.0)
                val = p_min + frac * (p_max - p_min)
                return f"${fmt.format(val)}$", val
        return "N/A", 0.0

## 1. Full-set retrieval table (`tab_full_set.tex`)

This table summarises the ring geometry, nuisance, and derived parameters for the
all-free retrieval where $(p, f_e, i_R, \theta_R, \alpha, \rho_{\star,\mathrm{true}}, b)$ are all
sampled freely.

In [4]:
RUN_SUFFIX = (
    "NS_exorings_kde_delta-T14-T23-rho_obs_nlive1200_dlogz0.01"
    "_NKDE5000_seed2026_rhoFREE_bFREE_alphaFREE_pFREE_rhoMasuda"
)

meta = {}
for pl in PLANETS:
    meta_file = FULL_RESULTS / f"{CASE}_{pl}_{RUN_SUFFIX}_meta.json"
    with open(meta_file) as f:
        meta[pl] = json.load(f)
    print(f"Loaded meta for planet {pl}")

params = [
    ("fe", "{:.2f}"),
    ("ir", "{:.1f}"),
    ("theta", "{:.1f}"),
    ("p", "{:.4f}"),
    ("alpha", "{:.3f}"),
    ("rho_true", "{:.2f}"),
    ("b", "{:.3f}")
]

vals = []
for param, fmt in params:
    vb, ub, lb = get_stats(meta["b"], param)
    vd, ud, ld = get_stats(meta["d"], param)
    vals.append(format_val(vb, ub, lb, fmt))
    vals.append(format_val(vd, ud, ld, fmt))

Rb, rhob = get_derived(meta["b"])
Rd, rhod = get_derived(meta["d"])
vals.extend([Rb, Rd, rhob, rhod])

latex_full = r"""\begin{table}[t]
\centering
\footnotesize
\caption{Full-set retrieval results for \exoplanet{Kepler-51}{b} and \exoplanet{Kepler-51}{d}. The table reports the median values and the 16th and 84th percentiles for the ring geometry parameters, nuisance parameters, and selected derived physical properties. We assume a stellar radius of $0.869\,R_\odot$ \citep{Berger2023} and a planetary mass of $6.9\,M_\oplus$ for both planets \citep{Masuda2024}. For comparison, under a ringless model, the inferred planetary radii are $6.85\,R_\oplus$ (\exoplanet{Kepler-51}{b}) and $9.36\,R_\oplus$ (\exoplanet{Kepler-51}{d}), yielding anomalous densities of $0.11\,\mathrm{g\,cm^{-3}}$ and $0.038\,\mathrm{g\,cm^{-3}}$, respectively.}
\label{tab:full_set_results}
\setlength{\tabcolsep}{3pt} %% Default is 6pt
\renewcommand{\arraystretch}{1.3}
\begin{tabular*}{\columnwidth}{@{\extracolsep{\fill}}lcc}
\hline
\hline
Parameter & \exoplanet{Kepler-51}{b} & \exoplanet{Kepler-51}{d} \\
\hline
\multicolumn{3}{c}{\textit{Ring geometry}} \\
$f_e$ [$R_p$] & %s & %s \\
$i_R$ [$^\circ$] & %s & %s \\
$\theta_R$ [$^\circ$] & %s & %s \\
$p$ & %s & %s \\
$\alpha$ & %s & %s \\
\hline
\multicolumn{3}{l}{\textit{Nuisance parameters}} \\
$\rho_{\star,\mathrm{true}}$ [$\mathrm{g\,cm^{-3}}$] & %s & %s \\
$b$ & %s & %s \\
\midrule
\multicolumn{3}{l}{\textit{Derived parameters}} \\
$R_p$ [$R_\oplus$] & %.2f & %.2f \\
$\rho_p$ [$\mathrm{g\,cm^{-3}}$] & %.3f & %.3f \\
\hline
\hline
\end{tabular*}
\end{table}
"""

result = latex_full % tuple(vals)

out_path = _REPO / "papers" / "kepler51" / "tab_full_set.tex"
with open(out_path, "w") as f:
    f.write(result + "\n")
print(f"Written: {out_path}")
print()
print(result)

Loaded meta for planet b
Loaded meta for planet d
Written: /Users/jzuluaga/dev/PRisma/papers/kepler51/tab_full_set.tex

\begin{table}[t]
\centering
\footnotesize
\caption{Full-set retrieval results for \exoplanet{Kepler-51}{b} and \exoplanet{Kepler-51}{d}. The table reports the median values and the 16th and 84th percentiles for the ring geometry parameters, nuisance parameters, and selected derived physical properties. We assume a stellar radius of $0.869\,R_\odot$ \citep{Berger2023} and a planetary mass of $6.9\,M_\oplus$ for both planets \citep{Masuda2024}. For comparison, under a ringless model, the inferred planetary radii are $6.85\,R_\oplus$ (\exoplanet{Kepler-51}{b}) and $9.36\,R_\oplus$ (\exoplanet{Kepler-51}{d}), yielding anomalous densities of $0.11\,\mathrm{g\,cm^{-3}}$ and $0.038\,\mathrm{g\,cm^{-3}}$, respectively.}
\label{tab:full_set_results}
\setlength{\tabcolsep}{3pt} % Default is 6pt
\renewcommand{\arraystretch}{1.3}
\begin{tabular*}{\columnwidth}{@{\extracolsep{\fil

## 2. Grid-search Golden Sample table (`tab_grid_golden.tex`)

This table collects the "Golden Sample" retrievals from the radius–opacity grid search.

In [5]:
base_dir = EXPLORE_RESULTS
rows = []

for planet in PLANETS:
    score_file = base_dir / f"scoring_{CASE}_{planet}.json"
    if not score_file.exists():
        print(f"WARNING: {score_file} not found, skipping planet {planet}")
        continue
    with open(score_file) as f:
        scores = json.load(f)

    for score in scores:
        if score.get("category") == "[Excellent] Golden Sample":
            tag = score["tag"]
            meta_file = base_dir / f"{tag}_meta.json"
            with open(meta_file) as f:
                m = json.load(f)

            p_str, p_val = get_param(m, "p", tag, "{:.4f}")
            R_p_earth = p_val * R_star_earth
            p_earth_str = "{:.2f}".format(R_p_earth)
            p_comb = f"{p_str}\\;({p_earth_str})"

            fe_str, fe_val = get_param(m, "fe", tag, "{:.2f}")
            ir_str, _ = get_param(m, "ir", tag, "{:.1f}")
            theta_str, _ = get_param(m, "theta", tag, "{:.1f}")
            alpha_str, _ = get_param(m, "alpha", tag, "{:.2f}")
            rho_star_str, _ = get_param(m, "rho_true", tag, "{:.2f}")
            b_str, _ = get_param(m, "b", tag, "{:.2f}")

            rho_p = RHO_E * M_p_earth / (R_p_earth**3) if R_p_earth > 0 else 0
            rho_p_str = f"{rho_p:.3f}"

            lnZ = score.get("logz", 0.0)
            lnZ_str = f"{lnZ:.2f}"

            import urllib.parse
            base_url = ("https://github.com/seap-udea/PRisma/blob/master/"
                        "pipeline/kepler_51/results/exorings/explore_radius_alpha_masuda/figures/")
            pdf_fname = (f"{CASE}_{planet}_cat1_GoldenSample_"
                         f"{score.get('zkey', '')}-{tag}_corner.png")
            ppc_fname = (f"{CASE}_{planet}_cat1_GoldenSample_"
                         f"{score.get('zkey', '')}-{tag}_ppc.png")
            pdf_url = base_url + urllib.parse.quote(pdf_fname)
            ppc_url = base_url + urllib.parse.quote(ppc_fname)
            links_str = (f"\\href{{{pdf_url}}}{{pdf}} "
                         f"\\textbar \\href{{{ppc_url}}}{{ppc}}")

            rows.append(dict(
                planet=planet, p_comb=p_comb, fe=fe_str, ir=ir_str,
                theta=theta_str, alpha=alpha_str, rho_star=rho_star_str,
                b=b_str, rho_p=rho_p_str, lnZ=lnZ, lnZ_str=lnZ_str,
                links_str=links_str, p_val=p_val, fe_val=fe_val,
            ))

# Sort by planet then lnZ descending
rows.sort(key=lambda x: (x["planet"], -x["lnZ"], -x["p_val"], -x["fe_val"]))
counts = {"b": sum(1 for r in rows if r["planet"] == "b"),
          "d": sum(1 for r in rows if r["planet"] == "d")}

print(f"Golden Sample: {counts.get('b', 0)} rows for b, {counts.get('d', 0)} rows for d")

Golden Sample: 6 rows for b, 6 rows for d


In [6]:
latex = []
latex.append("\\begin{table*}[t]")
latex.append("\\centering")
latex.append("\\footnotesize")
latex.append("\\caption{Golden Sample retrievals from the radius-alpha grid search "
             "for \\exoplanet{Kepler-51}{b} and \\exoplanet{Kepler-51}{d}.}")
latex.append("\\label{tab:grid_golden}")
latex.append("\\setlength{\\tabcolsep}{4pt}")
latex.append("\\begin{tabular*}{\\textwidth}"
             "{@{\\extracolsep{\\fill}} c @{\\hspace{0.3em}} c @{\\hspace{0.3em}} "
             "c @{\\hspace{0.3em}} c @{\\hspace{0.3em}} c @{\\hspace{0.3em}} "
             "c @{\\hspace{0.3em}} c @{\\hspace{0.3em}} c @{\\hspace{0.3em}} "
             "c @{\\hspace{0.3em}} c @{\\hspace{0.3em}} c @{}}")
latex.append("\\toprule")
latex.append("& \\multicolumn{7}{c}{Input selection} & Other & Metrics & Joint pdf \\\\")
latex.append("\\cmidrule(lr){2-8} \\cmidrule(lr){9-9} \\cmidrule(lr){10-10} \\cmidrule(lr){11-11}")
latex.append("Planet & $p\\;[R_\\star]\\;(R_\\oplus)$ & $f_e\\;[R_p]$ & "
             "$i_R\\;[^\\circ]$ & $\\theta_R\\;[^\\circ]$ & $\\alpha$ & "
             "$\\rho_{\\star,\\mathrm{true}}\\:[\\mathrm{g\\,cm^{-3}}]$ & $b$ & "
             "$\\rho_p\\:[\\mathrm{g\\,cm^{-3}}]$ & $\\ln \\mathcal{Z}$ & Plot link \\\\")
latex.append("\\midrule")

current_planet = None
for r in rows:
    if r["planet"] != current_planet:
        if current_planet is not None:
            latex.append("\\midrule")
        current_planet = r["planet"]
        pl_name = f"\\exoplanet{{Kepler-51}}{{{current_planet}}}"
        count = counts[current_planet]
        planet_col = f"\\multirow{{{count}}}{{*}}{{\\rotatebox{{90}}{{{pl_name}}}}}"
    else:
        planet_col = ""

    row_str = (f"{planet_col} & {r['p_comb']} & {r['fe']} & {r['ir']} & "
               f"{r['theta']} & {r['alpha']} & {r['rho_star']} & {r['b']} & "
               f"{r['rho_p']} & {r['lnZ_str']} & {r['links_str']} \\\\")
    latex.append(row_str)

latex.append("\\bottomrule")
latex.append("\\end{tabular*}")
latex.append("\\end{table*}")

result_grid = "\n".join(latex) + "\n"
out_path = _REPO / "papers" / "kepler51" / "tab_grid_golden.tex"
with open(out_path, "w") as f:
    f.write(result_grid)
print(f"Written: {out_path}")

Written: /Users/jzuluaga/dev/PRisma/papers/kepler51/tab_grid_golden.tex


## Done

Both tables have been regenerated.